# NEXUS LLM Extension: Skill Generation + Training

End-to-end run of the `nexus_continuous.llm` pipeline against the NEXUS
trainer for one environment:

1. Build prompts from `envs/env_registry.py` metadata.
2. Call an `LLMClient` (Mock backend by default so this runs anywhere with
   no API access. Switch `BACKEND` below to `"hf"` or `"openai"` for a real model).
3. Validate + compile the returned JSON into a runnable JAX policy module
   and sanity-check it on synthetic obs.
4. Train it with `run_llm_experiment()`, which wires
   `USE_LLM_SKILLS` / `LLM_SKILLSET` / `OBS_FIELDS` into the training config.
5. Save the skill JSON + checkpoint under `results/`.
6. Run the interactive refinement loop:
   propose -> train (real trainer) -> summarize -> 
   feed metrics back to the LLM -> revise, for several iterations. 

In [1]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

ENV_NAME = "CartpoleBalance"     
CONFIG = "configs/cartpole_balance_neural.yaml"
BACKEND = "hf" # "mock" | "hf" | "openai"
SEED = 0

## 1-2. Build the LLM client and generate a skillset

In [ ]:
from nexus_continuous.envs.env_registry import ENV_REGISTRY
from nexus_continuous.llm.client import LLMClient, LLMConfig, MockSkillGenerator
from nexus_continuous.llm.pipeline import generate_skillset

meta = ENV_REGISTRY[ENV_NAME]
print(meta["task"])
print("fields:", meta["fields"])

if BACKEND == "hf":
    client = LLMClient(
        LLMConfig(backend="hf", seed=SEED),
        mock_generator=MockSkillGenerator(meta["fields"], seed=SEED)
    )
else:
    client = LLMClient(LLMConfig(backend=BACKEND))

skillset = generate_skillset(
    env_name=ENV_NAME,
    observation_schema="\n".join(meta["fields"]),
    task_description=meta["task"],
    client=client,
    allowed_fields=set(meta["fields"]),
)
for s in skillset.skills:
    print(f"- {s.name}: activation_rule={s.activation_rule!r}, {len(s.reward_terms)} reward term(s)")

## 3. Compile and sanity-check with the interpreter

In [ ]:
from dataclasses import asdict
import numpy as np
import jax.numpy as jnp
from nexus_continuous.llm.interpreter import make_policy_module

policy_module = make_policy_module(asdict(skillset), field_names=tuple(meta["fields"]))
print("Skills:", policy_module.SKILL_NAMES)

rng = np.random.default_rng(0)
obs = jnp.asarray(rng.standard_normal((8, len(meta["fields"]))).astype("float32"))
action = jnp.asarray(rng.standard_normal((8, 2)).astype("float32"))
done = jnp.zeros((8,), dtype=bool)

rewards = policy_module.skill_rewards(obs, obs, action, None, done, None)
mask = policy_module.skill_mask(obs, None)
print("skill_rewards shape:", rewards.shape, "| skill_mask shape:", mask.shape)
print(policy_module.explain_policy())

## 4. Training

In [ ]:
from nexus_continuous.scripts.run_llm_experiment import run_llm_experiment

result = run_llm_experiment(
    env_name=ENV_NAME,
    config=CONFIG,
    seed=SEED,
    output="results",
    client=client,
)
print("Trained skills:", result["skillset"].skills and [s.name for s in result["skillset"].skills])

In [ ]:
import jax

metrics = jax.device_get(result["metrics"])
eval_metrics = jax.device_get(result["eval_metrics"])
print("final env_reward_mean:", float(np.asarray(metrics["returns/env_reward_mean"])[-1]))
if eval_metrics:
    print("deterministic eval:", {k: float(np.asarray(v)) for k, v in eval_metrics.items()
                                   if k in ("episode_return_mean", "primary_success_rate")})

## 5. Interactive refinement loop

In [ ]:
import jax
from nexus_continuous.llm.pipeline import LLMSkillPipeline
from nexus_continuous.llm.refinement_loop import LLMRefinementLoop, RefinementConfig, summarize_metrics
from nexus_continuous.utils import load_config
from nexus_continuous.algorithms import hierarchical_ac_pqn_playground as algo

NUM_ITERATIONS = 3

def real_train_fn(skillset):
    """train_fn(skillset) -> metrics, backed by the real trainer."""
    ss_dict = asdict(skillset) if hasattr(skillset, "__dataclass_fields__") else skillset
    cfg = load_config(CONFIG)
    cfg["ENV_NAME"] = ENV_NAME
    cfg["SEED"] = SEED
    cfg["USE_LLM_SKILLS"] = True
    cfg["LLM_SKILLSET"] = ss_dict
    cfg["OBS_FIELDS"] = tuple(meta["fields"])
    out = algo.run_training(cfg)
    m = jax.device_get(out.metrics)
    return {
        "returns/env_reward_mean": float(np.asarray(m["returns/env_reward_mean"])[-1]),
        "returns/skill_reward_mean": float(np.asarray(m["returns/skill_reward_mean"])[-1]),
        "policy_diag/primary_success_rate": float(np.nanmean(
            np.asarray(m.get("policy_diag/primary_success_rate", 0.0))[-5:])),
        "policy_diag/primary_goal_metric": float(np.nanmean(
            np.asarray(m.get("policy_diag/primary_goal_metric", 0.0))[-5:])),
    }

pipeline = LLMSkillPipeline(client)
loop = LLMRefinementLoop(pipeline, client)
refine_cfg = RefinementConfig(
    env_name=ENV_NAME,
    observation_schema="\n".join(meta["fields"]),
    task_description=meta["task"],
    num_iterations=NUM_ITERATIONS,
    allowed_fields=set(meta["fields"]),
)
refine_result = loop.run(refine_cfg, real_train_fn)
for rec in refine_result.history:
    print(f"iter {rec.iteration}: {summarize_metrics(rec.metrics)}"
          + ("" if rec.refinement_ok else f"  [refinement failed: {rec.refinement_error}]"))

In [ ]:
from nexus_continuous.llm.plot import plot_refinement
import matplotlib.pyplot as plt
import os

os.makedirs("plots", exist_ok=True)
path = plot_refinement(refine_result.history, "plots/refinement_real.png",
                        title=f"{ENV_NAME} refinement loop (real trainer)")
plt.figure(figsize=(9, 4)); plt.imshow(plt.imread(path)); plt.axis("off"); plt.show()

print("Final skillset:", [s.name for s in refine_result.final_skillset.skills],
      "| stopped_early:", refine_result.stopped_early)